# 5 — Récursion, pile d’appels et mémoïsation


## Fil conducteur

Les chapitres précédents avançaient dans une séquence avec des boucles. Certains
problèmes se décrivent plus naturellement par une version plus petite d'eux-mêmes.
Nous construirons cette idée en séparant le cas qui s'arrête, l'appel plus petit,
la remontée du résultat et enfin la mémorisation des calculs répétés.




*La pile conserve les appels en attente ; le dictionnaire conserve des résultats réutilisables.*


## 1. Une fonction qui s’appelle elle-même
La récursion décrit un problème au moyen d'une version plus petite de lui-même.
Pour compter jusqu'à zéro, on passe de n à n−1. Deux éléments sont indispensables :
un **cas de base** dont la réponse est immédiate, et une mesure qui diminue vers ce cas.
Sans réduction effective, l'exécution n'atteint pas sa condition d'arrêt.

### Exemple illustré — un appel crée un appel plus petit

```text
compte(3)
  └─ compte(2)
       └─ compte(1)
            └─ compte(0)  ← arrêt
```

Le dernier appel n’en crée pas d’autre : c’est le cas de base qui arrête la
descente.


### Prouver une fonction récursive

Une trace explique un exemple ; une preuve couvre toutes les entrées du contrat.
On raisonne sur la mesure qui rapproche du cas de base.

1. **Base** : vérifier directement la réponse pour la plus petite mesure.
2. **Hypothèse** : supposer que l'appel sur une mesure plus petite respecte son
   contrat.
3. **Étape** : montrer que le travail local combiné à cette réponse construit la
   bonne réponse courante.
4. **Terminaison** : montrer qu'une mesure entière non négative diminue strictement.

Pour `factorielle(n)`, la base `n <= 1` rend 1. Si `factorielle(n-1)` rend bien
le produit de 1 à `n-1`, le multiplier par `n` donne le produit de 1 à `n`.
Comme `n` baisse d'une unité, la base finit par être atteinte.


## 2. Descendre n’est pas calculer toutes les réponses
Pour `factorielle(3)`, on attend d'abord la réponse de factorielle(2), qui attend
celle de factorielle(1). Le cas de base rend 1 ; les appels suspendus reprennent :
2×1=2, puis 3×2=6. `return` rend la réponse à **l'appelant**, pas à tous les appels.

| Appel actif | Travail restant |
|---|---|
| factorielle(3) | multiplier la réponse de factorielle(2) par 3 |
| factorielle(2) | multiplier la réponse de factorielle(1) par 2 |
| factorielle(1) | renvoyer 1 |

### Exemple illustré — descendre d’abord, calculer ensuite

```text
factorielle(3)
= 3 × factorielle(2)      attente
= 3 × 2 × factorielle(1)  attente
= 3 × 2 × 1               remontée
= 6
```

Pendant la descente, les multiplications restent en attente. Les résultats sont
produits pendant la remontée.




*Les appels attendent pendant la descente ; les valeurs sont calculées pendant la remontée.*


## 3. La pile d’appels dans la RAM
Chaque appel possède son contexte : paramètres, variables locales et position
à laquelle reprendre. Ces contextes forment la pile d'appels du processus Python.
Une profondeur n exige O(n) contextes ; une version itérative peut parfois
n'utiliser que O(1) variables. Python limite la profondeur récursive et ne transforme
pas automatiquement la récursion terminale en boucle. Pour des entrées profondes,
une pile explicite ou une boucle est souvent préférable.

### Exemple illustré — chaque appel possède son cadre

```text
haut de pile  │ n=1, retour à factorielle(2)
              │ n=2, retour à factorielle(3)
bas de pile   │ n=3, retour à l’appelant
```

Un cadre mémorise les variables locales et l’endroit où reprendre. Il disparaît
quand l’appel rend son résultat.


## 4. Le cache de mémoïsation est un autre objet
Un `memo = {}` est un dictionnaire Python en RAM. Il conserve une association
entrée → résultat. Ce n'est ni un fichier ni une zone spéciale appelée cache CPU.
Si `memo` n'a plus de référence, ses données peuvent être libérées ; un cache
conservé globalement ou par un décorateur peut vivre plus longtemps.
Le cache ne remplace pas la pile : une fonction récursive mémoïsée possède les deux.

### Exemple illustré — le cache se remplit une seule fois

| Demande | Action | Cache après l’action |
|---|---|---|
| `fibo(2)` | calculer | `{2: 1}` |
| `fibo(3)` | réutiliser 2, calculer 3 | `{2: 1, 3: 2}` |
| `fibo(2)` à nouveau | lire directement | inchangé |

Le cache vit dans un dictionnaire séparé de la pile d’appels.


## 5. Répéter des sous-problèmes ou explorer des choix ?
Fibonacci naïf répète les mêmes calculs : la mémoïsation les partage.
Le backtracking explore des décisions : on essaie, on poursuit, on annule.
Tous les problèmes récursifs ne bénéficient pas de la mémoïsation ; la factorielle
n'a pas de sous-problèmes répétés dans un appel unique.

### Exemple illustré — répétition ou choix ne sont pas la même chose

| Situation | Arbre observé | Outil utile |
|---|---|---|
| `fibo(5)` redemande `fibo(3)` | même sous-problème répété | mémoïsation |
| former tous les mots `AA`, `AB`, `BA`, `BB` | solutions différentes | backtracking |

Le cache évite un calcul déjà connu ; le backtracking explore volontairement des
branches différentes.


### Relations de récurrence et arbre d'appels

On peut décrire le coût par une relation. Une factorielle effectue un seul appel
plus petit et un travail local constant : `T(n) = T(n-1) + O(1)`, donc `T(n)=O(n)`.

Fibonacci naïf crée au contraire deux branches :

```text
fibo(5)
├── fibo(4)
│   ├── fibo(3)
│   └── fibo(2)
└── fibo(3)       ← sous-problème déjà calculé
```

Le nombre d'appels croît exponentiellement. La mémoïsation ne rend pas chaque appel
gratuit : elle garantit que chaque **état distinct** est développé une seule fois.
Ici il existe seulement `n+1` états et un nombre constant de dépendances par état,
d'où O(n) temps sous le modèle arithmétique choisi.

Une version itérative « de bas en haut » peut calculer les mêmes états sans pile
d'appels et ne conserver que les deux dernières valeurs. Mémoïsation et tabulation
sont deux organisations différentes d'une même réutilisation des sous-problèmes.
La clé du cache doit encoder tout ce qui influence la réponse ; sinon deux états
différents seraient confondus.




*Les nœuds répétés ont le même argument : un cache permet de développer chacun une seule fois.*

## Une trace descend puis remonte

Prédis l’ordre des affichages avant de lancer.


In [1]:
def factorielle(n, profondeur=0):
    if n < 0:
        raise ValueError("n doit être non négatif")
    print("  "*profondeur + f"entrée n={n}")
    if n <= 1:
        print("  "*profondeur + "retour 1")
        return 1
    resultat = n * factorielle(n-1, profondeur+1)
    print("  "*profondeur + f"retour {resultat}")
    return resultat
assert factorielle(3) == 6


entrée n=3
  entrée n=2
    entrée n=1
    retour 1
  retour 2
retour 6


### Trace visuelle : la descente empile, la remontée calcule

| Moment | Appels encore en attente | Valeur qui revient |
|---|---|---:|
| descente | `factorielle(3)` | — |
| descente | `factorielle(3) → factorielle(2)` | — |
| descente | `factorielle(3) → factorielle(2) → factorielle(1)` | — |
| cas de base | mêmes appels | 1 |
| remontée | `factorielle(3) → factorielle(2)` | `2 × 1 = 2` |
| remontée | `factorielle(3)` | `3 × 2 = 6` |

L’appel extérieur ne peut terminer qu’après avoir reçu le résultat de l’appel plus
petit. La pile conserve précisément ce travail en attente.

Dans le modèle scolaire où les multiplications coûtent O(1), temps O(n), pile O(n). Les très grands entiers modifient ce modèle ; les impressions ne sont pas incluses dans cette analyse.


## Voir le dictionnaire se remplir

Passe le même dictionnaire aux appels récursifs.


In [2]:
def fibo_memo(n, memo=None):
    if n < 0:
        raise ValueError("n négatif")
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n in memo:
        print("lecture cache :", n, "->", memo[n])
        return memo[n]
    memo[n] = fibo_memo(n-1, memo) + fibo_memo(n-2, memo)
    print("stockage :", n, "->", memo[n])
    return memo[n]
memo = {}
assert fibo_memo(5, memo) == 5
print("cache en RAM :", memo)


stockage : 2 -> 1
stockage : 3 -> 2
lecture cache : 2 -> 1
stockage : 4 -> 3
lecture cache : 3 -> 2
stockage : 5 -> 5
cache en RAM : {2: 1, 3: 2, 4: 3, 5: 5}


`memo=None` crée un cache neuf si l’appelant n’en fournit pas. Un défaut `memo={}` serait partagé entre appels. Fibonacci mémoïsé calcule O(n) états, avec O(n) entrées et O(n) profondeur dans le modèle à additions constantes. Il garde aussi des appels de lecture du cache.


## Exercices différenciés

Essaie chaque exercice avant de consulter le corrigé. Pour lancer les tests d’un exercice, remplace son `pass`, puis enlève le `#` devant les appels de test. Les cellules incomplètes restent exécutables.


### Palier A — Compte à rebours

Renvoie [n,n−1,…,0] pour un entier n non négatif. Version récursive demandée.

<details><summary>Indice progressif</summary>

Pour ne pas recopier une liste à chaque niveau, ajoute dans une liste partagée construite localement.

</details>


In [3]:
def compte_rebours(n):
    if n < 0:
        raise ValueError("n négatif")
    resultat = []
    def visiter(k):
        resultat.append(k)
        if k > 0:
            visiter(k-1)
    visiter(n)
    return resultat


In [4]:
assert compte_rebours(3) == [3,2,1,0]
assert compte_rebours(0) == [0]


**Pourquoi cette solution fonctionne**

O(n) en temps, O(n) pile et O(n) sortie. `[n] + compte_rebours(n-1)` copierait des listes de tailles croissantes et coûterait O(n²).


### Palier B — Somme récursive sans tranches

Renvoie la somme d’une liste en utilisant un indice qui avance. Évite `valeurs[1:]`.

<details><summary>Indice progressif</summary>

Le cas de base est un indice égal à la longueur.

</details>


In [5]:
def somme_recursive(valeurs):
    def visiter(i):
        if i == len(valeurs):
            return 0
        return valeurs[i] + visiter(i+1)
    return visiter(0)


In [6]:
assert somme_recursive([2,-3,7]) == 6
assert somme_recursive([]) == 0


**Pourquoi cette solution fonctionne**

O(n) temps, O(n) pile. Les tranches répétées auraient créé O(n²) copies d’éléments au total. Ici une boucle serait plus simple et économiserait la pile.


### Palier C — Escalier avec cache visible

On peut monter 1 ou 2 marches. Renvoie le nombre de façons d’atteindre n marches (n≥0). Pour n=0, une possibilité : ne faire aucun pas.

<details><summary>Indice progressif</summary>

ways(n)=ways(n−1)+ways(n−2), avec ways(0)=ways(1)=1.

</details>


In [7]:
def escalier(n):
    if n < 0:
        raise ValueError("n négatif")
    memo = {0:1, 1:1}
    def visiter(k):
        if k not in memo:
            memo[k] = visiter(k-1) + visiter(k-2)
        return memo[k]
    return visiter(n)


In [8]:
assert escalier(0) == 1
assert escalier(3) == 3
assert escalier(5) == 8


**Pourquoi cette solution fonctionne**

La dernière marche vient de n−1 ou n−2 ; ces cas sont disjoints et couvrent toutes les possibilités. O(n) états, O(n) pile et cache, sous hypothèse d’arithmétique constante.


## Questions de compréhension

1. Quelles sont les quatre parties d'une preuve récursive ?
2. Pourquoi Fibonacci naïf est-il coûteux alors que sa profondeur est seulement O(n) ?
3. Que doit contenir une clé de mémoïsation ?
4. Pourquoi une boucle peut-elle être préférable à une récursion pourtant correcte ?


**Éléments de réponse.** Il faut une base, une hypothèse sur une mesure plus petite,
une étape de construction et une mesure qui garantit la terminaison. Fibonacci crée
un nombre exponentiel d'appels répétés malgré une profondeur linéaire. La clé encode
tout l'état influençant la réponse. Une boucle évite la limite et la mémoire de la pile.


## À retenir et vérifier

Une récursion correcte précise cas de base, réduction et réponse retournée. Trace à la fois les appels et les retours. Mémoïsation et pile ont des rôles différents.

**Autoévaluation :** peux-tu expliquer l'état des variables sur un petit exemple,
justifier la condition d'arrêt et donner un cas limite ? Peux-tu distinguer
l'espace de travail de l'espace occupé par les résultats ?

**Débrief en binôme :** présente une première solution correcte, puis un compromis
(vitesse, mémoire, lisibilité). Une optimisation doit préserver le contrat.


**Étape suivante.** Le chapitre 6 utilisera la récursion non plus pour un seul sous-problème, mais pour explorer plusieurs choix possibles.
